In [ ]:
import anndata as ad
# adata = ad.read_h5ad("./data/larry/postprocessed.h5ad")
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 1. Expression matrix (X)
# ----------------------------
X = adata.layers["spliced"]
hvg_mask = adata.var["highly_variable"].values
X = X[:, hvg_mask]

if hasattr(X, "toarray"):
    X = X.toarray()

X = np.log1p(X)

X = StandardScaler(
    with_mean=True,
    with_std=True
).fit_transform(X)

# ----------------------------
# 2. Velocity matrix (V)
# ----------------------------
V = adata.layers["velocity"]
V = V[:, hvg_mask]

if hasattr(V, "toarray"):
    V = V.toarray()

# impute NaNs gene-wise
gene_means = np.nanmean(V, axis=0)
gene_means[np.isnan(gene_means)] = 0.0

nan_mask = np.isnan(V)
if nan_mask.any():
    V[nan_mask] = gene_means[nan_mask.nonzero()[1]]

# scale variance per gene (no centering)
V = StandardScaler(
    with_mean=False,
    with_std=True
).fit_transform(V)
X.shape, V.shape

In [ ]:
# rng = np.random.default_rng(0)
# n, d = X.shape

# k_per_point = 200

# # --------------------------------------------------
# # Sample random pairs
# # --------------------------------------------------
# i_idx = np.repeat(np.arange(n), k_per_point)
# j_idx = rng.integers(0, n, size=n * k_per_point)

# mask = i_idx != j_idx
# i_idx = i_idx[mask]
# j_idx = j_idx[mask]

# # --------------------------------------------------
# # Compute raw distances ONCE
# # --------------------------------------------------
# diff = X[i_idx] - X[j_idx]        # (N_pairs, d)
# d_raw = np.linalg.norm(diff, axis=1)
data = np.load("./data/larry/d_raw_pairs.npz")

i_idx = data["i_idx"]
j_idx = data["j_idx"]
d_raw = data["d_raw"]

i_idx.shape, j_idx.shape, d_raw.shape

In [ ]:
def distance_corr(d_ref, Z, i_idx, j_idx):
    d_Z = np.linalg.norm(Z[i_idx] - Z[j_idx], axis=1)
    return pearsonr(d_ref, d_Z)[0]

In [ ]:
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
from sklearn.manifold import TSNE
import umap

results = {
    "PCA": {},
    "UMAP": {},
    "tSNE": {},
    "PCA+TPS": {},
    "UMAP+TPS": {},
    "tSNE+TPS": {},
}

embeddings = {
    "PCA": {},
    "UMAP": {},
    "tSNE": {},
}

pca_dims = [2, 4, 8, 16, 32, 64]
for d in pca_dims:
    print(f"[PCA] Computing {d}D embedding...")
    
    Z = PCA(n_components=d, random_state=0).fit_transform(X)
    
    embeddings["PCA"][d] = Z
    results["PCA"][d] = distance_corr(d_raw, Z, i_idx, j_idx)
    
    print(f"    Pearson corr = {results['PCA'][d]:.4f}")

umap_dims = [2, 4, 8]
umap_kwargs = dict(
    n_neighbors=30,
    min_dist=0.5,
    metric="euclidean",
    random_state=0,
)

for d in umap_dims:
    print(f"[UMAP] Computing {d}D embedding...")
    
    Z = umap.UMAP(
        n_components=d,
        **umap_kwargs
    ).fit_transform(X)
    
    embeddings["UMAP"][d] = Z
    results["UMAP"][d] = distance_corr(d_raw, Z, i_idx, j_idx)
    print(f"    Pearson corr = {results['UMAP'][d]:.4f}")


tsne_dims = [2]   # strongly recommend 2D only
for d in tsne_dims:
    print(f"[t-SNE] Computing {d}D embedding (this may take a while)...")
    
    Z = TSNE(
        n_components=d,
        perplexity=30,
        init="pca",
        learning_rate="auto",
        random_state=0,
        max_iter=1000,     # <-- FIX HERE
        verbose=1,         # progress output
    ).fit_transform(X)
    
    embeddings["tSNE"][d] = Z
    results["tSNE"][d] = distance_corr(d_raw, Z, i_idx, j_idx)
    
    print(f"    Pearson corr = {results['tSNE'][d]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

dims = np.array(sorted(results["PCA"].keys()))
corrs = np.array([results["PCA"][d] for d in dims])

plt.figure(figsize=(4.5, 3.5))

plt.plot(
    dims,
    corrs,
    marker="o",
    linewidth=2,
    color="black",
)

plt.xscale("log", base=2)
plt.xticks(dims, dims)

plt.xlabel("PCA embedding dimension")
plt.ylabel("Global distance correlation (Pearson)")
plt.ylim(0, 1.0)

plt.title("PCA baseline: metric fidelity vs dimension", fontsize=11)

plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
d

In [ ]:
from scripts.VectorFieldEmbedder import VectorFieldEmbedder

results_tps = {
    "PCA": {},
    "UMAP": {},
    "tSNE": {},
}

embeddings_tps = {
    "PCA": {},
    "UMAP": {},
    "tSNE": {},
}

for method, emb_dict in embeddings.items():
    for d, Z in emb_dict.items():
        print(f"[TPS] {method}-{d}D: fitting TPS...")
        if d < 30:
            emb = VectorFieldEmbedder(
                X,
                V,
                X_emb=Z,
                dof=50,
                use_PCA=True,
                pca_components=30,
                max_tps_points=4000,
            )
            emb.initialize_embedding(42)
            # TPS is fitted during initialization / embedding setup
            # (assuming VectorFieldEmbedder does this internally)
    
            X_tps = emb.tps.predict(Z)
    
            embeddings_tps[method][d] = X_tps
    
            results_tps[method][d] = distance_corr(d_raw, X_tps, i_idx, j_idx)
    
            print(f"    TPS Pearson corr = {results_tps[method][d]:.4f}")

In [ ]:
# def binned_corr(d_ref, d_test, n_bins=10, n_per_bin=5000, rng=np.random.default_rng(0)):
#     qs = np.linspace(0, 1, n_bins + 1)
#     edges = np.quantile(d_ref, qs)

#     corrs = []
#     centers = []

#     for b in range(n_bins):
#         lo, hi = edges[b], edges[b + 1]
#         idx = np.where((d_ref >= lo) & (d_ref <= hi))[0]

#         if len(idx) == 0:
#             corrs.append(np.nan)
#             centers.append((lo + hi) / 2)
#             continue

#         take = min(n_per_bin, len(idx))
#         sub = rng.choice(idx, size=take, replace=False)

#         corrs.append(pearsonr(d_ref[sub], d_test[sub])[0])
#         centers.append(np.median(d_ref[sub]))

#     return np.array(centers), np.array(corrs)

# def embedding_dist(Z, i_idx, j_idx):
#     """
#     Compute pairwise distances for a given embedding or reconstruction Z.
#     """
#     return np.linalg.norm(Z[i_idx] - Z[j_idx], axis=1)


# rng = np.random.default_rng(0)

# curves = {}   # name -> (centers, corr)

# # ---------- PCA ----------
# for d, Z in embeddings["PCA"].items():
#     if d <= 16:
#         print(f"PCA-{d}: computing distances")
#         d_test = embedding_dist(Z, i_idx, j_idx)
    
#         print(f"PCA-{d}: computing binned correlation")
#         centers, c = binned_corr(d_raw, d_test, rng=rng)
#         curves[f"PCA-{d}"] = (centers, c)
    
#         print(f"PCA-{d}+TPS: computing distances")
#         Z_tps = embeddings_tps["PCA"][d]
#         d_test_tps = embedding_dist(Z_tps, i_idx, j_idx)
    
#         print(f"PCA-{d}+TPS: computing binned correlation")
#         centers, c = binned_corr(d_raw, d_test_tps, rng=rng)
#         curves[f"PCA-{d}+TPS"] = (centers, c)


# # ---------- UMAP ----------
# for d, Z in embeddings["UMAP"].items():
#     print(f"UMAP-{d}: computing distances")
#     d_test = embedding_dist(Z, i_idx, j_idx)

#     print(f"UMAP-{d}: computing binned correlation")
#     centers, c = binned_corr(d_raw, d_test, rng=rng)
#     curves[f"UMAP-{d}"] = (centers, c)

#     print(f"UMAP-{d}+TPS: computing distances")
#     Z_tps = embeddings_tps["UMAP"][d]
#     d_test_tps = embedding_dist(Z_tps, i_idx, j_idx)

#     print(f"UMAP-{d}+TPS: computing binned correlation")
#     centers, c = binned_corr(d_raw, d_test_tps, rng=rng)
#     curves[f"UMAP-{d}+TPS"] = (centers, c)


# # ---------- t-SNE ----------
# for d, Z in embeddings["tSNE"].items():
#     print(f"tSNE-{d}: computing distances")
#     d_test = embedding_dist(Z, i_idx, j_idx)

#     print(f"tSNE-{d}: computing binned correlation")
#     centers, c = binned_corr(d_raw, d_test, rng=rng)
#     curves[f"tSNE-{d}"] = (centers, c)

#     print(f"tSNE-{d}+TPS: computing distances")
#     Z_tps = embeddings_tps["tSNE"][d]
#     d_test_tps = embedding_dist(Z_tps, i_idx, j_idx)

#     print(f"tSNE-{d}+TPS: computing binned correlation")
#     centers, c = binned_corr(d_raw, d_test_tps, rng=rng)
#     curves[f"tSNE-{d}+TPS"] = (centers, c)



import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 5))

for name, (centers, corr) in curves.items():
    if "+TPS" in name:
        plt.plot(
            centers,
            corr,
            linewidth=2.2,
            alpha=0.9,
            label=name,
        )
    else:
        plt.plot(
            centers,
            corr,
            linestyle="--",
            linewidth=1.5,
            alpha=0.6,
            label=name,
        )

plt.xlabel("Raw distance (bin center)")
plt.ylabel("Pearson correlation\nwithin distance bin")
# plt.ylim(-0.2, 1.0)

plt.legend(
    frameon=False,
    fontsize=8,
    ncol=3,
)

plt.title("All embeddings and TPS reconstructions (unfiltered)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
centers, centers_base

In [ ]:
delta_curves = {}   # name -> (centers, delta_corr)

for name, (centers, corr) in curves.items():
    if "+TPS" not in name:
        continue

    base_name = name.replace("+TPS", "")
    if base_name not in curves:
        continue

    centers_base, corr_base = curves[base_name]

    # sanity check (optional but good)
    # assert np.allclose(centers, centers_base)

    delta = corr - corr_base
    delta_curves[base_name] = (centers, delta)


import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4.5))

for name, (centers, delta) in delta_curves.items():
    if name.startswith("PCA"):
        style = "--"
        alpha = 0.7
    elif name.startswith("UMAP"):
        style = "-"
        alpha = 0.9
    else:  # tSNE
        style = "-"
        alpha = 0.9

    plt.plot(
        centers,
        delta,
        linestyle=style,
        linewidth=2,
        alpha=alpha,
        label=name,
    )

plt.axhline(0, color="black", linewidth=1, alpha=0.5)

plt.xlabel("Raw distance (bin center)")
plt.ylabel("Δ Pearson correlation\n(TPS − embedding)")
plt.ylim(-0.1, 0.4)

plt.legend(frameon=False, fontsize=8, ncol=2)
plt.title("TPS gain in metric fidelity across distance scales", fontsize=11)

plt.tight_layout()
plt.show()